# Fine-tune test: Qwen/Qwen3-0.6B

Four versions of the same pretrained model: untouched (few-shot), full fine-tune, full fine-tune + state slots, LoRA + state slots. Scored on planning in the world (clean, messy, filler), honest gaps, English kept, Keal's real messages, and again after squashing to 4 bits. A pilot runs every code path first. Results: `FT_RESULTS_qwen3-0.6b.md`.

In [ ]:
import base64, os, subprocess, sys
for f, b in {"ft.py": "...", "messages.json": "..."}.items():
    open(f, "wb").write(base64.b64decode(b))
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import transformers
v = tuple(int(x) for x in transformers.__version__.split(".")[:2])
print("transformers", transformers.__version__)
if v < (4, 51):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.51", "datasets"], check=True)
    print("upgraded transformers; restart not needed since workers are fresh processes")

In [ ]:
import glob, json, time, torch
MODEL = "Qwen/Qwen3-0.6B"
NGPU = max(1, torch.cuda.device_count())
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
PLAN = {0: ["untouched", "ft"], 1: ["ft-slots", "lora-slots"]} if NGPU > 1 else {0: ["untouched", "ft", "ft-slots", "lora-slots"]}


def run(tag):
    procs = []
    for g, versions in PLAN.items():
        log = open(f"{tag}_gpu{g}.txt", "w")
        cmd = " && ".join(f"{sys.executable} -u ft.py worker --model {MODEL} --version {v} --gpu {g} --tag {tag}" for v in versions)
        procs.append((g, subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT)))
    t0 = time.time()
    while any(p.poll() is None for _, p in procs):
        time.sleep(60 if tag == "pilot" else 300)
        for g, _ in procs:
            lines = [l for l in open(f"{tag}_gpu{g}.txt").read().splitlines() if l.strip()]
            print(f"[{(time.time()-t0)/60:5.1f} min gpu{g}] " + (lines[-1][:200] if lines else "starting"), flush=True)
    ok = all(p.returncode == 0 for _, p in procs)
    for g, p in procs:
        if p.returncode != 0:
            print(f"--- gpu{g} failed (exit {p.returncode}) ---\n" + open(f"{tag}_gpu{g}.txt").read()[-5000:])
    return ok

In [ ]:
ok = run("pilot")
done = sorted(glob.glob("pilot_*.json"))
lines = ["## Pilot (20 training steps, 2 problems per cell: every code path once)", ""]
for f in done:
    r = json.load(open(f))
    lines.append(f"- {r['version']}: ran in {r['total_s']}s, clean short {r['plans']['clean/short']['success']*100:.0f}%")
GO = ok and len(done) == 4
lines += ["", "Pilot passed; full run started." if GO else "**Pilot failed; full run skipped. See pilot_gpu*.txt.**"]
open("PILOT.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))

In [ ]:
if GO:
    run("full")
else:
    print("skipped")

In [ ]:
r = subprocess.run([sys.executable, "ft.py", "report", "--model", MODEL], capture_output=True, text=True)
print(r.stdout[-20000:], r.stderr[-3000:])